# Punto 4: BFS: 8-puzzle

In [1]:
# Caso sin solución: la paridad de inversiones no coincide con la del objetivo.
estado_sin_solucion = (
    1, 2, 3,
    4, 5, 0,
    6, 8, 7,
)

# Caso con solución: este tablero está a 31 movimientos, la distancia máxima del 8-puzzle.
estado_largo = (
    8, 6, 7,
    2, 5, 4,
    3, 0, 1,
)

# Configuración que queremos alcanzar en ambos casos.
estado_objetivo = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0,
)

estado_sin_solucion, estado_largo, estado_objetivo

((1, 2, 3, 4, 5, 0, 6, 8, 7), (8, 6, 7, 2, 5, 4, 3, 0, 1), (1, 2, 3, 4, 5, 6, 7, 8, 0))


In [2]:
def sucesores_8_puzzle(estado):
    # Convertimos la tupla a lista porque necesitamos intercambiar posiciones.
    estado = list(estado)

    # Buscamos dónde está la casilla vacía.
    indice_cero = estado.index(0)

    # Convertimos el índice lineal 0..8 en fila y columna de una matriz 3x3.
    fila, columna = divmod(indice_cero, 3)

    # Posibles desplazamientos de la casilla vacía:
    # arriba, abajo, izquierda y derecha.
    movimientos = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1),
    ]

    # Aquí almacenaremos los tableros alcanzables con un solo movimiento.
    sucesores = []

    for df, dc in movimientos:
        # Calculamos la nueva posición candidata del espacio vacío.
        nf, nc = fila + df, columna + dc

        # El movimiento es válido solamente si permanece dentro del tablero.
        if 0 <= nf < 3 and 0 <= nc < 3:
            # Convertimos otra vez fila/columna a un índice lineal.
            nuevo_indice = nf * 3 + nc

            # Copiamos el tablero para no modificar el estado original.
            nuevo = estado.copy()

            # Intercambiamos el 0 con la ficha adyacente.
            nuevo[indice_cero], nuevo[nuevo_indice] = (
                nuevo[nuevo_indice],
                nuevo[indice_cero],
            )

            # Los estados se representan como tuplas para poder usarlos
            # dentro de un set de visitados.
            sucesores.append(tuple(nuevo))

    return sucesores


sucesores_8_puzzle(estado_sin_solucion)

[(1, 2, 0, 4, 5, 3, 6, 8, 7), (1, 2, 3, 4, 5, 7, 6, 8, 0), (1, 2, 3, 4, 0, 5, 6, 8, 7)]


In [3]:
from collections import deque


def accion_entre(anterior, siguiente):
    # El nombre de la acción es el desplazamiento de la casilla vacía.
    origen = anterior.index(0)
    destino = siguiente.index(0)
    df = (destino // 3) - (origen // 3)
    dc = (destino % 3) - (origen % 3)
    return {
        (-1, 0): "Arriba",
        (1, 0): "Abajo",
        (0, -1): "Izquierda",
        (0, 1): "Derecha",
    }[(df, dc)]


def bfs_8_puzzle(estado_inicial, estado_objetivo):
    """
    Búsqueda primero en anchura para el 8-puzzle.

    Devuelve el camino, el número de movimientos y los estados explorados.
    Si no hay solución, estados y movimientos quedan en None.
    """
    estado_inicial = tuple(estado_inicial)
    estado_objetivo = tuple(estado_objetivo)
    visitados = set()
    cola = deque()
    padre = {}

    cola.append(estado_inicial)
    visitados.add(estado_inicial)
    padre[estado_inicial] = None

    while cola:
        actual = cola.popleft()
        if actual == estado_objetivo:
            estados = []
            nodo = actual
            while nodo is not None:
                estados.append(nodo)
                nodo = padre[nodo]
            estados.reverse()
            acciones = [
                accion_entre(estados[i], estados[i + 1])
                for i in range(len(estados) - 1)
            ]
            return {
                "estados": estados,
                "acciones": acciones,
                "movimientos": len(acciones),
                "explorados": len(visitados),
            }

        for sucesor in sucesores_8_puzzle(actual):
            if sucesor not in visitados:
                visitados.add(sucesor)
                padre[sucesor] = actual
                cola.append(sucesor)

    return {
        "estados": None,
        "acciones": [],
        "movimientos": None,
        "explorados": len(visitados),
    }


solucion_sin_solucion = bfs_8_puzzle(estado_sin_solucion, estado_objetivo)
solucion_larga = bfs_8_puzzle(estado_largo, estado_objetivo)


In [4]:
def mostrar_solucion_8_puzzle(solucion_8_puzzle):
    if solucion_8_puzzle is None or solucion_8_puzzle["estados"] is None:
        print("No se encontró solución.")
        if solucion_8_puzzle is not None:
            print("Movimientos: -")
            print("Estados explorados:", solucion_8_puzzle["explorados"])
        return

    # Extraemos los estados y acciones encontrados por BFS.
    estados = solucion_8_puzzle["estados"]
    acciones = solucion_8_puzzle["acciones"]

    print("Estado inicial:", estados[0])

    # Cada acción lleva al estado ubicado en la siguiente posición.
    for i, accion in enumerate(acciones, start=1):
        print(f"{i}. {accion:15s} -> {estados[i]}")

    print("\nMovimientos:", solucion_8_puzzle["movimientos"])
    print("Estados explorados:", solucion_8_puzzle["explorados"])

In [5]:
print("Caso sin solución")
mostrar_solucion_8_puzzle(solucion_sin_solucion)

print("\nCaso con 31 movimientos")
mostrar_solucion_8_puzzle(solucion_larga)


Caso sin solución
No se encontró solución.
Movimientos: -
Estados explorados: 181440

Caso con 31 movimientos
Estado inicial: (8, 6, 7, 2, 5, 4, 3, 0, 1)
1. Arriba          -> (8, 6, 7, 2, 0, 4, 3, 5, 1)
2. Arriba          -> (8, 0, 7, 2, 6, 4, 3, 5, 1)
3. Izquierda       -> (0, 8, 7, 2, 6, 4, 3, 5, 1)
4. Abajo           -> (2, 8, 7, 0, 6, 4, 3, 5, 1)
5. Abajo           -> (2, 8, 7, 3, 6, 4, 0, 5, 1)
6. Derecha         -> (2, 8, 7, 3, 6, 4, 5, 0, 1)
7. Derecha         -> (2, 8, 7, 3, 6, 4, 5, 1, 0)
8. Arriba          -> (2, 8, 7, 3, 6, 0, 5, 1, 4)
9. Arriba          -> (2, 8, 0, 3, 6, 7, 5, 1, 4)
10. Izquierda       -> (2, 0, 8, 3, 6, 7, 5, 1, 4)
11. Abajo           -> (2, 6, 8, 3, 0, 7, 5, 1, 4)
12. Izquierda       -> (2, 6, 8, 0, 3, 7, 5, 1, 4)
13. Abajo           -> (2, 6, 8, 5, 3, 7, 0, 1, 4)
14. Derecha         -> (2, 6, 8, 5, 3, 7, 1, 0, 4)
15. Derecha         -> (2, 6, 8, 5, 3, 7, 1, 4, 0)
16. Arriba          -> (2, 6, 8, 5, 3, 0, 1, 4, 7)
17. Arriba          -> (2, 6, 0, 5, 3, 